# ConditionalSNN Unified Test

This notebook performs an end-to-end smoke test for `MixedGAMRegressor`/`ConditionalSNN` by:
1. Loading a manageable slice of the California Housing dataset.
2. Registering a custom torchmetrics metric (Spearman rank correlation).
3. Training the model with group labels and monitoring multiple metrics/losses.
4. Evaluating predictions on the validation split.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = project_root.parent
if project_root.exists() and str(project_root) not in sys.path:
    sys.path.append(str(project_root))
print(f"Using project root: {project_root}")


Using project root: D:\Code


In [2]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import pandas as pd

dataset = fetch_california_housing(as_frame=True)
frame = dataset.frame
X = frame[dataset.feature_names]
y = frame['MedHouseVal']
groups = pd.cut(
    frame['Latitude'],
    bins=5,
    labels=[f'lat_{idx}' for idx in range(5)],
    include_lowest=True,
)

(
    X_train,
    X_val,
    y_train,
    y_val,
    g_train,
    g_val,
) = train_test_split(
    X,
    y,
    groups,
    test_size=0.2,
    random_state=42,
    stratify=groups,
)
print(f'Train shape: {X_train.shape}, Val shape: {X_val.shape}')


Train shape: (16512, 8), Val shape: (4128, 8)


In [3]:
from torchmetrics import SpearmanCorrCoef
from src.models.conditional_sensitivity_neural_network import register_metric

register_metric('spearman', lambda: SpearmanCorrCoef())
print('Registered custom Spearman metric')


Using project root: D:\Code
Registered custom Spearman metric


In [8]:
from src.models import ConditionalSNNRegressor

model = ConditionalSNNRegressor(
    mode='group_residual',
    metric_names=("rmse", "mae", "spearman", "bias"),
    n_epochs=20,
    batch_size=128,
    learning_rate=5e-4,
    patience=200,
    verbose=True,
)

model.fit(
    X_train,
    y_train,
    groups=g_train,
    X_val=X_val,
    y_val=y_val,
    groups_val=g_val,
    progress_refresh_rate=5,
)
print('Training history (last 5):', model.training_history[-5:])


Seed set to 42
d:\Code\.venv\Lib\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: Metric `SpearmanCorrcoef` will save all targets and predictions in the buffer. For large datasets, this may lead to large memory footprint.
  warnings.warn(*args, **kwargs)
GPU available: False, used: False
TPU available: False, using: 0 TPU cores


[MixedGAM] Training mode=group_residual via PyTorch Lightning...


d:\Code\.venv\Lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:751: Checkpoint directory D:\Code\notebooks\checkpoints exists and is not empty.


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

d:\Code\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
d:\Code\.venv\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:433: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=20` reached.


[MixedGAM] Finished training. Last loss=0.48010
Training history (last 5): [0.35024625062942505, 0.24871312081813812, 0.35048383474349976, 0.2699005603790283, 0.48009735345840454]


In [9]:
from sklearn.metrics import root_mean_squared_error, r2_score

preds = model.predict(X_val, groups=g_val)
rmse = root_mean_squared_error(y_val, preds)
r2 = r2_score(y_val, preds)
print(f'Validation RMSE: {rmse:.4f}')
print(f'Validation R^2: {r2:.4f}')


Validation RMSE: 0.5642
Validation R^2: 0.7586


### Baseline KPI

In [5]:
from catboost import CatBoostRegressor, Pool
from sklearn.metrics import root_mean_squared_error, r2_score

train_pool = Pool(X_train, y_train, cat_features=None)
val_pool = Pool(X_val, y_val)

cb = CatBoostRegressor(
    depth=6,
    learning_rate=0.05,
    iterations=500,
    loss_function="RMSE",
    verbose=False,
)
cb.fit(train_pool, eval_set=val_pool)

val_preds = cb.predict(val_pool)
rmse = root_mean_squared_error(y_val, val_preds)
print(f"CatBoost baseline RMSE: {rmse:.4f}")
r2 = r2_score(y_val, val_preds)
print(f'Validation R^2: {r2:.4f}')

CatBoost baseline RMSE: 0.4498
Validation R^2: 0.8466


### Full Config Example

In [ ]:
from src.models import ConditionalSNNRegressor

gam = ConditionalSNNRegressor(
    mode="group_residual",
    base_hidden_units=(128, 64, 32),
    base_activation="tanh",
    base_dropout=0.1,
    base_norm="layernorm",
    residual_hidden_units=(64, 32),
    residual_activation="relu",
    residual_dropout=0.15,
    residual_norm="layernorm",
    parallel_feature_mlp=True,
    learning_rate=7e-4,
    weight_decay=1e-4,
    batch_size=512,
    n_epochs=250,
    patience=60,
    lambda_center=1e-3,
    lambda_group_l1=5e-5,
    lambda_group_l2=5e-5,
    lambda_orth=5e-5,
    lambda_base_target=2e-4,
    residual_contribution_l1=5e-5,
    residual_contribution_l2=5e-5,
    metric_names=("rmse", "mae", "spearman", "bias"),
    verbose=True,
)
gam.fit(X_train,
        y_train,
        groups=g_train,
        X_val=X_val,
        y_val=y_val,
        groups_val=g_val)


In [16]:
preds = gam.predict(X_val, groups=g_val)
rmse = root_mean_squared_error(y_val, preds)
r2 = r2_score(y_val, preds)
print(f'Validation RMSE: {rmse:.4f}')
print(f'Validation R^2: {r2:.4f}')

Validation RMSE: 0.5074
Validation R^2: 0.8047


### Tuning Result

In [ ]:
from ray import tune
from src.models.conditional_sensitivity_neural_network import ConditionalSNNRegressor

search = ConditionalSNNRegressor.ray_tune_search(
    X_train,
    y_train,
    groups=g_train,
    num_samples=40,
    metric="val_loss",
    mode="min",
    param_space={
        "learning_rate": tune.loguniform(3e-4, 2e-3),
        "base_hidden_units": tune.choice([(96, 48), (128, 64, 32)]),
        "residual_hidden_units": tune.choice([(48, ), (64, 32)]),
        "base_dropout": tune.uniform(0.05, 0.2),
        "residual_dropout": tune.uniform(0.1, 0.3),
        "residual_norm": tune.choice(["layernorm", "none"]),
        "lambda_group_l1": tune.loguniform(1e-5, 5e-4),
        "lambda_group_l2": tune.loguniform(1e-5, 5e-4),
        "lambda_orth": tune.loguniform(1e-5, 5e-4),
        "batch_size": tune.choice([256, 512]),
        "mode": tune.choice(["group_residual", "group_affine"]),
    },
)

best_cfg = search["best_config"]
best_model = ConditionalSNNRegressor(**best_cfg)
best_model.fit(
    X_train,
    y_train,
    groups=g_train,
    X_val=X_val,
    y_val=y_val,
    groups_val=g_val,
)

2025-11-10 22:27:31,603	ERROR tune_controller.py:1331 -- Trial task failed for trial _trainable_605cd_00001
Traceback (most recent call last):
  File "d:\Code\.venv\Lib\site-packages\ray\air\execution\_internal\event_manager.py", line 110, in resolve_future
    result = ray.get(future)
             ^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\client_mode_hook.py", line 104, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\worker.py", line 2961, in get
    values, debugger_breakpoint = worker.get_objects(
                                  ^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\worker.py", line 1028, in get_objects
    raise value
ray.exceptions.ActorDiedError: The actor died unexpectedly 

(raylet) Traceback (most recent call last):
  File "python\\ray\\_raylet.pyx", line 1982, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 2093, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 1989, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 1932, in ray._raylet.execute_task.function_executor
  File "d:\Code\.venv\Lib\site-packages\ray\_private\function_manager.py", line 693, in actor_method_executor
    return method(__ray_actor, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\function_manager.py", line 610, in temporary_actor_method
    raise RuntimeError(
RuntimeError: The actor with name ImplicitFunc failed to import on the worker. This may be because needed library dependencies are not installed in the worker environment:

Traceback (most recent call last):
  File "d:\Code\.venv\Lib\site-packages\ray\_private\function_manager.py", line 649, in _load

2025-11-10 22:27:31,666	ERROR tune_controller.py:1331 -- Trial task failed for trial _trainable_605cd_00000
Traceback (most recent call last):
  File "d:\Code\.venv\Lib\site-packages\ray\air\execution\_internal\event_manager.py", line 110, in resolve_future
    result = ray.get(future)
             ^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\auto_init_hook.py", line 22, in auto_init_wrapper
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\client_mode_hook.py", line 104, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\worker.py", line 2961, in get
    values, debugger_breakpoint = worker.get_objects(
                                  ^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\worker.py", line 1028, in get_objects
    raise value
ray.exceptions.ActorDiedError: The actor died unexpectedly 

(raylet) Traceback (most recent call last):
  File "python\\ray\\_raylet.pyx", line 1982, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 2093, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 1989, in ray._raylet.execute_task
  File "python\\ray\\_raylet.pyx", line 1932, in ray._raylet.execute_task.function_executor
  File "d:\Code\.venv\Lib\site-packages\ray\_private\function_manager.py", line 693, in actor_method_executor
    return method(__ray_actor, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Code\.venv\Lib\site-packages\ray\_private\function_manager.py", line 610, in temporary_actor_method
    raise RuntimeError(
RuntimeError: The actor with name ImplicitFunc failed to import on the worker. This may be because needed library dependencies are not installed in the worker environment:

Traceback (most recent call last):
  File "d:\Code\.venv\Lib\site-packages\ray\_private\function_manager.py", line 649, in _load

2025-11-10 22:27:39,770	WARNING tune.py:219 -- Stop signal received (e.g. via SIGINT/Ctrl+C), ending Ray Tune run. This will try to checkpoint the experiment state one last time. Press CTRL+C (or send SIGINT/SIGKILL/SIGTERM) to skip. 
2025-11-10 22:27:39,825	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to 'C:/Users/USER/ray_results/_trainable_2025-11-10_22-27-18' in 0.0516s.
2025-11-10 22:27:40,341	ERROR tune.py:1037 -- Trials did not complete: [_trainable_605cd_00000, _trainable_605cd_00001, _trainable_605cd_00002, _trainable_605cd_00003, _trainable_605cd_00004, _trainable_605cd_00005, _trainable_605cd_00006, _trainable_605cd_00007]
2025-11-10 22:27:40,341	INFO tune.py:1041 -- Total run time: 15.64 seconds (14.97 seconds for the tuning loop).
2025-11-10 22:27:40,344	WARNING tune.py:1056 -- Experiment has been interrupted, but the most recent state was saved.
Resume experiment with: Tuner.restore(path="C:/Users/USER/ray_results/_trainable_2025-

RuntimeError: No best trial found for the given metric: val_loss. This means that no trial has reported this metric, or all values reported for this metric are NaN. To not ignore NaN values, you can set the `filter_nan_and_inf` arg to False.

(pid=gcs_server) [2025-11-10 22:27:51,477 E 18124 21280] (gcs_server.exe) gcs_server.cc:302: Failed to establish connection to the event+metrics exporter agent. Events and metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14
(raylet) [2025-11-10 22:27:53,840 E 22548 19764] (raylet.exe) main.cc:975: Failed to establish connection to the metrics exporter agent. Metrics will not be exported. Exporter agent status: RpcError: Running out of retries to initialize the metrics agent. rpc_code: 14


: 